# agent_learning — notebook smoke test

Confirms Jupyter runs smoothly in the local `.venv` with GPU access.

**Kernel:** pick **`Python (agent_learning, RTX5090)`** in the top-right kernel picker.

Run the cells top to bottom (Shift+Enter).

In [1]:
# 1) Environment + GPU
import sys, platform, torch
print("python :", sys.version.split()[0], "|", platform.system())
print("venv   :", sys.executable)
print("torch  :", torch.__version__, "| cuda", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")
print("sm_%d%d" % torch.cuda.get_device_capability(0), "| bf16", torch.cuda.is_bf16_supported())

python : 3.12.13 | Windows
venv   : F:\agent_learning\.venv\Scripts\python.exe
torch  : 2.11.0+cu128 | cuda 12.8
GPU    : NVIDIA GeForce RTX 5090
sm_120 | bf16 True


In [2]:
# 2) A quick GPU matmul (sanity + speed)
import time
n = 8192
a = torch.randn(n, n, device="cuda", dtype=torch.bfloat16)
b = torch.randn(n, n, device="cuda", dtype=torch.bfloat16)
for _ in range(3):
    c = a @ b
torch.cuda.synchronize()
t0 = time.time()
for _ in range(20):
    c = a @ b
torch.cuda.synchronize()
dt = (time.time() - t0) / 20
print(f"{n}x{n} bf16 matmul: {dt*1e3:.2f} ms -> {2*n**3/dt/1e12:.0f} TFLOP/s")

8192x8192 bf16 matmul: 4.60 ms -> 239 TFLOP/s


In [3]:
# 3) ipywidgets progress bar (interactive UI works in-notebook)
import ipywidgets as widgets
from IPython.display import display
bar = widgets.IntProgress(value=0, min=0, max=10, description="warmup:")
display(bar)
for i in range(11):
    bar.value = i
print("ipywidgets OK")

IntProgress(value=0, description='warmup:', max=10)

ipywidgets OK


In [4]:
# 4) Load the tokenizer of a local model (fast, no weights) to prove HF stack works
from pathlib import Path
from transformers import AutoTokenizer
mpath = Path.cwd().parent / "LLM_model_weights" / "Qwen3-14B"
tok = AutoTokenizer.from_pretrained(str(mpath))
print("loaded tokenizer:", mpath.name, "| vocab", tok.vocab_size)
print("tokens:", tok("Hello, agentic world!")["input_ids"])

loaded tokenizer: Qwen3-14B | vocab 151643
tokens: [9707, 11, 933, 4256, 1879, 0]


If all four cells ran without errors, Jupyter is working smoothly on the GPU.

Tip: to run a full model generation in a notebook, see `scripts/example_inference.py` — you can
import and call it, or paste its body into a cell. For long jobs, keep an eye on VRAM with `nvitop` in a terminal.